# Archived team notebook
See [workflow and limitations](README.md) before running. Outputs and session metadata have been cleared. Supply your own data paths and checkpoints. These notebooks were not rerun during packaging.

# Stable Diffusion v1.5 + LoRA 訓練流程

In [ ]:
# STEP 1: 安裝必要套件
# 包含 diffusers（Stable Diffusion 套件）、peft（LoRA 微調）、transformers（CLIP 模型）、safetensors 等
!pip install -q diffusers accelerate peft transformers safetensors bitsandbytes

In [ ]:
# STEP 2: 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 3: 解壓上傳的 .zip 資料集
import zipfile
zip_path = "/content/drive/MyDrive/lora_crack_dataset/lora_crack_dataset.zip"  ### 替換成你自己的路徑
extract_path = "/content/lora_dataset"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
# STEP 4: 載入 Stable Diffusion 模型與 VAE 編碼器
# 並加上 LoRA 結構，只微調 UNet 中的 Attention 層權重。
from diffusers import StableDiffusionPipeline, AutoencoderKL
from peft import get_peft_model, LoraConfig
import torch

base_model = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    base_model,
    torch_dtype=torch.float16
).to("cuda")

# VAE 也載入（雖然部分流程可省略，但保留以供手動 encode）
vae = AutoencoderKL.from_pretrained(
    base_model,
    subfolder="vae",
    torch_dtype=torch.float32  # ✔️ 完整精度，避免精度與輸入不一致
).to("cuda")

# 啟用 gradient checkpointing（降低記憶體使用量）
pipe.unet.enable_gradient_checkpointing()


# 設定 LoRA 微調哪些 Attention 子模組
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "to_q", "to_k", "to_v", "to_out.0",  # UNet 基礎模組
        "query", "key", "value", "proj_attn"  # 有些 diffusers 模型的變形結構會用到
    ]
)

pipe.unet = get_peft_model(pipe.unet, lora_config)

In [ ]:
# ==== STEP 5 (REPLACED): 建立 Dataset（多副檔名 + 遞迴 + 同名 .txt 配對）====
import os
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 如果你是解壓到 /content/lora_dataset，保留不變
extract_path = "/content/lora_dataset"   # <--- 仍用你的設定

root = Path(extract_path)
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

train_data = []
missing_txt = []
for p in root.rglob("*"):                        # 遞迴子資料夾
    if p.suffix.lower() in valid_exts:
        txt = p.with_suffix(".txt")
        if txt.exists():
            train_data.append((str(p), str(txt)))
        else:
            missing_txt.append(str(p.name))

print(f"✅ 成功載入 {len(train_data)} 筆訓練資料（圖片 + prompt）")
if missing_txt:
    print(f"⚠️ 有 {len(missing_txt)} 張圖片缺少對應 .txt（僅列前 10 筆）：")
    for name in missing_txt[:10]:
        print("  -", name)

# 解析度：與推理一致建議 512 或 640
RES = 512
to_tensor = transforms.Compose([
    transforms.Resize((RES, RES), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],[0.5, 0.5, 0.5])  # 三通道 [-1,1]
])

class LoRADataset(Dataset):
    def __init__(self, pairs):
        self.items = pairs

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        img_path, txt_path = self.items[idx]
        img = Image.open(img_path).convert("RGB")
        img = to_tensor(img)
        text = Path(txt_path).read_text(encoding="utf-8").strip()
        return {"pixel_values": img, "text": text}

dataset = LoRADataset(train_data)
# 顯存不夠可改 batch_size=1；我們用梯度累積來維持等效批量
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
print("📦 batch_size =", dataloader.batch_size)

In [ ]:
# ==== STEP 6: LoRA 訓練（裂縫強化版，支援新版 torch.autocast / GradScaler）====
import math, random, torch
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- 建議超參數 ----
EPOCHS = 16              # 至少 12–20
LR = 1e-5                # 比 1e-4 更穩定
ACCUM_STEPS = 2          # 等效總 batch = dataloader.batch_size * ACCUM_STEPS
TEXT_DROPOUT_P = 0.1     # 10% 空字串（classifier-free guidance）
USE_P2 = True            # 啟用 p2-like loss
P2_GAMMA = 0.5
NOISE_OFFSET = 0.02
MIXED_PREC = "fp16"      # "fp16" 或 "bf16"

pipe.unet.train()
pipe.unet.enable_gradient_checkpointing()

optimizer = torch.optim.AdamW(
    pipe.unet.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=1e-2
)

MAX_TRAIN_STEPS = EPOCHS * math.ceil(len(dataloader) / ACCUM_STEPS)
num_warmup_steps = int(0.05 * MAX_TRAIN_STEPS)
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=MAX_TRAIN_STEPS
)

# ⚡ 新版 API
scaler = torch.amp.GradScaler("cuda", enabled=(MIXED_PREC=="fp16"))

# ---- caption 裂縫詞強化 ----
def emphasize_crack(text: str) -> str:
    boosters = ["crack", "fracture", "surface crack", "fracture line"]
    extras = ", ".join(random.sample(boosters, k=random.choice([1,1,2])))
    return (text + ", " + extras).strip(", ")

alphas_cumprod = pipe.scheduler.alphas_cumprod.to(device)
global_step = 0

for epoch in range(EPOCHS):
    pbar = tqdm(dataloader, desc=f"epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(pbar):
        images = batch["pixel_values"].to(device, non_blocking=True)
        texts  = batch["text"]

        # 10% 空字串 + 裂縫詞強化
        proc_texts = []
        for t in texts:
            if random.random() < TEXT_DROPOUT_P:
                proc_texts.append("")
            else:
                proc_texts.append(emphasize_crack(t))

        tok = pipe.tokenizer(
            proc_texts, padding="max_length", truncation=True,
            max_length=pipe.tokenizer.model_max_length, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            # VAE encode → latent
            latents = vae.encode(images.to(dtype=torch.float32)).latent_dist.sample()
            latents = latents * 0.18215
            if NOISE_OFFSET and NOISE_OFFSET > 0:
                latents = latents + NOISE_OFFSET * torch.randn_like(latents)

            # 文本嵌入
            text_embeds = pipe.text_encoder(tok["input_ids"])[0]

        bsz = latents.shape[0]
        timesteps = torch.randint(
            0, pipe.scheduler.config.num_train_timesteps, (bsz,), device=device
        ).long()

        noise = torch.randn_like(latents)
        noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)

        autocast_dtype = torch.float16 if MIXED_PREC=="fp16" else torch.bfloat16
        use_autocast = torch.cuda.is_available()
        with torch.autocast("cuda", dtype=autocast_dtype, enabled=use_autocast):
            noise_pred = pipe.unet(noisy_latents, timesteps, encoder_hidden_states=text_embeds).sample

            base_loss = (noise_pred - noise) ** 2
            if USE_P2:
                a = alphas_cumprod.clamp(min=1e-5, max=1.0)[timesteps]
                sigma2 = (1 - a) / a
                weights = (1 + sigma2) ** (-P2_GAMMA)
                loss = (weights.view(-1,1,1,1) * base_loss).mean()
            else:
                loss = base_loss.mean()

        scaler.scale(loss / ACCUM_STEPS).backward()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            lr_scheduler.step()
            global_step += 1

        pbar.set_postfix(loss=float(loss.detach().cpu()), lr=optimizer.param_groups[0]["lr"])

    # 可選：每個 epoch 存 checkpoint
    # torch.save(pipe.unet.state_dict(), f"/content/ckpt_unet_lora_epoch{epoch+1}.pt")


In [ ]:
# STEP 7：正確儲存 LoRA 權重為 .safetensors 格式
from peft import get_peft_model_state_dict
from safetensors.torch import save_file
import os

output_dir = "./"  # 或你指定的雲端資料夾
output_path = os.path.join(output_dir, "lora_crack_dataset.safetensors") ###自己要的檔名

# 只儲存 LoRA 權重（小且可通用）
save_file(get_peft_model_state_dict(pipe.unet), output_path)

print("✅ LoRA 訓練完成，只儲存 LoRA 權重為：", output_path)

In [ ]:
# 檢查 .safetensors 內容（可選）
from safetensors.torch import load_file

state_dict = load_file("lora_crack_dataset.safetensors") ###自己要的檔名
print(" 權重層級列表：")
for k in state_dict.keys():
    print(k)